# Datos geográficos de Mérida

Ejecuta las celdas desde arriba con **Run All**. Se requieren los cuatro ZIP originales en `data/raw`: Censo, Marco Geoestadístico, DENUE y ATUS.

Si falta GeoPandas, ejecuta `%pip install geopandas` en una celda y reinicia el kernel.


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
from IPython.display import display

# Buscar data/raw desde notebooks/ o desde la raíz del proyecto.
# Si tu carpeta está en otro lugar, escribe su ruta en DATA_RAW.
DATA_RAW = None
if DATA_RAW is None:
    candidates = [Path.cwd() / "data" / "raw"]
    candidates += [parent / "data" / "raw" for parent in Path.cwd().parents]
    DATA_RAW = next((p for p in candidates if p.is_dir()), None)
    if DATA_RAW is None:
        raise FileNotFoundError(
            f"No se encontró data/raw desde {Path.cwd()}. "
            "Abre el notebook dentro del proyecto o configura DATA_RAW."
        )
DATA_RAW = Path(DATA_RAW).expanduser().resolve()
print("Carpeta de datos:", DATA_RAW)

def data_file(name):
    path = DATA_RAW / name
    if not path.is_file():
        raise FileNotFoundError(f"Falta el archivo: {path}")
    return path

def zip_member(archive, basename):
    with zipfile.ZipFile(archive) as z:
        matches = [name for name in z.namelist()
                   if Path(name).name.lower() == basename.lower()]
    if len(matches) != 1:
        raise ValueError(f"Se esperaba un archivo {basename} en {archive.name}; encontrados: {matches}")
    return matches[0]

def geo_layer(archive, basename):
    member = zip_member(archive, basename)
    return gpd.read_file(f"zip://{archive.as_posix()}!{member}")


## Censo 2020 ageb

https://www.inegi.org.mx/programas/ccpv/2020/#datos_abiertos (Principales resultados por AGEB y manzana urbana)

In [ ]:
path_censo_zip = data_file("ageb_mza_urbana_31_cpv2020_csv.zip")
with zipfile.ZipFile(path_censo_zip) as z:
    for filename in z.namelist():
        if filename.lower().endswith(".csv"):
            print(filename)


In [ ]:
import pandas as pd
import zipfile

path_zip = path_censo_zip

with zipfile.ZipFile(path_zip, 'r') as z:
    # Busca automáticamente el archivo del conjunto de datos dentro del ZIP
    csv_target = [f for f in z.namelist() if f.endswith('.csv') and 'conjunto_de_datos' in f][0]
    
    with z.open(csv_target) as f:
        df_censo = pd.read_csv(f, encoding='utf-8-sig', low_memory=False,
                               dtype={'AGEB': 'string'})

df_censo.columns = df_censo.columns.str.strip()
df_censo["AGEB"] = df_censo["AGEB"].str.strip().str.upper().str.zfill(4)
for column in ["ENTIDAD", "MUN", "LOC", "MZA"]:
    df_censo[column] = pd.to_numeric(df_censo[column], errors="raise")
df_censo.head()

In [ ]:
df_censo.columns.tolist()

In [ ]:
[c for c in df_censo.columns if "AGEB" in c.upper()]

In [ ]:
[c for c in df_censo.columns if "POB" in c.upper()]

In [ ]:
# Filtrar registros de la ciudad de Mérida
censo_merida = df_censo[
    (df_censo["MUN"] == 50) &
    (df_censo["LOC"] == 1)
].copy()

print("Registros de Mérida:", len(censo_merida))

censo_merida[["MUN", "NOM_MUN", "LOC", "NOM_LOC", "AGEB", "MZA"]].head(10)

In [ ]:
censo_ageb = censo_merida[
    (censo_merida["NOM_LOC"] == "Total AGEB urbana") &
    (censo_merida["MZA"] == 0)
].copy()

print("AGEB en Censo:", len(censo_ageb))
print("AGEB únicos:", censo_ageb["AGEB"].nunique())

censo_ageb[
    ["MUN", "NOM_MUN", "LOC", "AGEB", "POBTOT", "POB0_14", "POB15_64", "POB65_MAS", "PEA"]
].head()

## Marco Geoestadístico. Censo de Población y Vivienda 2020

http://inegi.org.mx/app/biblioteca/ficha.html?upc=889463807469

GeoPandas debe estar instalado en el entorno del kernel. La instalación se realiza solo si hace falta, como se indica al inicio.


In [ ]:
import geopandas as gpd

print(gpd.__version__)

In [ ]:
path_geo_zip = data_file("31_yucatan.zip")
with zipfile.ZipFile(path_geo_zip) as z:
    shapefiles = [f for f in z.namelist() if f.lower().endswith(".shp")]
for shp in shapefiles:
    print(shp)


In [ ]:
import os
import pandas as pd
import geopandas as gpd

results = []

for shp in shapefiles:

    path_mapa = f"zip://{path_geo_zip.as_posix()}!{shp}"
    gdf = gpd.read_file(path_mapa)

    results.append({
        "file": os.path.basename(shp),
        "rows": len(gdf),
        "columns": len(gdf.columns),
        "crs": str(gdf.crs),
        "geometry": gdf.geometry.geom_type.unique().tolist(),
        "valid_geometries": gdf.geometry.is_valid.sum()
    })

layers_info = pd.DataFrame(results)
layers_info

In [ ]:
gdf_31ti = geo_layer(path_geo_zip, "31ti.shp")
gdf_31ti.head()


In [ ]:
gdf_31ti.columns.tolist()

In [ ]:
gdf_31ti.crs

In [ ]:
gdf_31ti.geometry.geom_type.value_counts()

In [ ]:
gdf_31ti.geometry.is_valid.value_counts()

In [ ]:
gdf_31m = geo_layer(path_geo_zip, "31m.shp")
gdf_31a = geo_layer(path_geo_zip, "31a.shp")

for layer in [gdf_31m, gdf_31a]:
    for column, width in {"CVE_MUN": 3, "CVE_LOC": 4, "CVE_AGEB": 4}.items():
        layer[column] = layer[column].astype("string").str.strip().str.upper().str.zfill(width)

print("31m")
print(gdf_31m.columns.tolist())
display(gdf_31m.head())
print("31a")
print(gdf_31a.columns.tolist())
display(gdf_31a.head())


In [ ]:
print("31m CRS:", gdf_31m.crs)
print("31m geometrías:", gdf_31m.geometry.geom_type.value_counts())
print("31m válidas:", gdf_31m.geometry.is_valid.value_counts())

print("\n31a CRS:", gdf_31a.crs)
print("31a geometrías:", gdf_31a.geometry.geom_type.value_counts())
print("31a válidas:", gdf_31a.geometry.is_valid.value_counts())

In [ ]:
gdf_31a["CVE_MUN"].value_counts()

In [ ]:
# Municipios disponibles en la capa 31a
print("Municipios:")
print(gdf_31a["CVE_MUN"].value_counts().sort_index())

# Verificar Mérida
print("\n¿Existe CVE_MUN = 050?")
print((gdf_31a["CVE_MUN"] == "050").any())

# Filtrar Mérida
ageb_merida = gdf_31a[
    gdf_31a["CVE_MUN"] == "050"
].copy()

print("\nNúmero de polígonos en Mérida:")
print(len(ageb_merida))

print("\nNúmero de AGEB diferentes:")
print(ageb_merida["CVE_AGEB"].nunique())

print("\nLocalidades dentro de Mérida:")
print(ageb_merida["CVE_LOC"].value_counts())

print("\nPrimeros registros:")
display(ageb_merida.head())

In [ ]:
ageb_merida["CVE_AGEB"].nunique()

In [ ]:
import matplotlib.pyplot as plt

ax = ageb_merida.plot(
    figsize=(10, 10),
    edgecolor="black",
    linewidth=0.3
)

ax.set_title("AGEB del municipio de Mérida")
ax.set_axis_off()

plt.show()

In [ ]:
ageb_merida_ciudad = gdf_31a[
    (gdf_31a["CVE_MUN"] == "050") &
    (gdf_31a["CVE_LOC"] == "0001")
].copy()

print("AGEB de la ciudad de Mérida:", len(ageb_merida_ciudad))

In [ ]:
ageb_cartografia = set(ageb_merida_ciudad["CVE_AGEB"])
ageb_censo = set(censo_ageb["AGEB"])

print("AGEB en cartografía:", len(ageb_cartografia))
print("AGEB en Censo:", len(ageb_censo))

print("AGEB en ambos:", len(ageb_cartografia & ageb_censo))
print("AGEB solo en cartografía:", len(ageb_cartografia - ageb_censo))
print("AGEB solo en Censo:", len(ageb_censo - ageb_cartografia))

In [ ]:
manzanas_merida = gdf_31m[
    (gdf_31m["CVE_MUN"] == "050") &
    (gdf_31m["CVE_LOC"] == "0001")
].copy()

print("Manzanas en Mérida:", len(manzanas_merida))

In [ ]:
manzanas_merida["AMBITO"].value_counts()

In [ ]:
ageb_urbanas = set(ageb_merida_ciudad["CVE_AGEB"])

ageb_manzanas = set(
    manzanas_merida["CVE_AGEB"]
)

print("AGEB en la cartografía:", len(ageb_urbanas))
print("AGEB con manzanas:", len(ageb_manzanas))

print(
    "AGEB sin manzanas:",
    len(ageb_urbanas - ageb_manzanas)
)

In [ ]:
import matplotlib.pyplot as plt

ax = ageb_merida_ciudad.plot(
    figsize=(10, 10),
    edgecolor="black",
    linewidth=0.3
)

ax.set_title("AGEB de la ciudad de Mérida")
ax.set_axis_off()

plt.show()

In [ ]:
print(ageb_merida_ciudad.crs)
print(ageb_merida_ciudad.geometry.geom_type.unique())
print(ageb_merida_ciudad.geometry.is_valid.all())
print(ageb_merida_ciudad.shape)
print(ageb_merida_ciudad["CVE_AGEB"].nunique())


## DENUE

https://www.inegi.org.mx/app/descarga/default.html 

In [ ]:
path_denue_zip = data_file("denue_31_csv.zip")
denue_csv = zip_member(path_denue_zip, "denue_inegi_31_.csv")
with zipfile.ZipFile(path_denue_zip) as z:
    print("Archivos dentro del ZIP:")
    for file in z.namelist():
        print(file)


In [ ]:
import pandas as pd
import zipfile

with zipfile.ZipFile(path_denue_zip) as z:
    with z.open(denue_csv) as f:
        denue_sample = pd.read_csv(f, nrows=5, encoding="latin-1")

print("Columns:", len(denue_sample.columns))
print(denue_sample.columns.tolist())

In [ ]:
with zipfile.ZipFile(path_denue_zip) as z:
    with z.open(denue_csv) as f:
        denue_df = pd.read_csv(
            f,
            encoding="latin-1",
            low_memory=False,
            dtype={"ageb": "string"}
        )

print("Records:", len(denue_df))
print("Columns:", len(denue_df.columns))
denue_df["ageb"] = denue_df["ageb"].str.strip().str.upper().str.zfill(4)
for column in ["cve_mun", "cve_loc"]:
    denue_df[column] = pd.to_numeric(denue_df[column], errors="raise")


In [ ]:
print(denue_df[["cve_mun", "municipio", "cve_loc", "localidad"]].head())
print(denue_df[["cve_mun", "cve_loc"]].dtypes)

In [ ]:
denue_merida = denue_df[
    (denue_df["cve_mun"] == 50) &
    (denue_df["cve_loc"] == 1)
].copy()

print("Establishments in Merida:", len(denue_merida))
print("Unique AGEBs:", denue_merida["ageb"].nunique())

denue_merida[
    ["id", "nom_estab", "codigo_act", "nombre_act",
     "ageb", "manzana", "latitud", "longitud"]
].head()

In [ ]:
ageb_denue = set(denue_merida["ageb"].dropna())
ageb_cartography = set(ageb_merida_ciudad["CVE_AGEB"].astype("string").str.zfill(4))

print("AGEBs in cartography:", len(ageb_cartography))
print("AGEBs in DENUE:", len(ageb_denue))
print("AGEBs in both:", len(ageb_cartography & ageb_denue))
print("AGEBs only in cartography:", len(ageb_cartography - ageb_denue))
print("AGEBs only in DENUE:", len(ageb_denue - ageb_cartography))

print("\nAGEB only in cartography:")
print(ageb_cartography - ageb_denue)

In [ ]:
print("AGEBs only in DENUE:")
print(sorted(ageb_denue - ageb_cartography))

print("\nAGEBs only in cartography:")
print(sorted(ageb_cartography - ageb_denue))

In [ ]:
denue_missing_ageb = denue_merida[
    denue_merida["ageb"].astype("string").str.zfill(4).isin(
        ageb_denue - ageb_cartography
    )
]

denue_missing_ageb[
    [
        "ageb",
        "nom_estab",
        "nombre_act",
        "municipio",
        "localidad",
        "latitud",
        "longitud"
    ]
].head(20)

In [ ]:
print(
    denue_missing_ageb["ageb"]
    .astype("string")
    .str.zfill(4)
    .value_counts()
    .sort_index()
)

In [ ]:
for file in sorted(DATA_RAW.rglob("*")):
    if file.is_file():
        print(file)


## ATUS

http://inegi.org.mx/programas/accidentes/#datos_abiertos 

In [ ]:
path_atus_zip = data_file("atus_2024_shp.zip")
with zipfile.ZipFile(path_atus_zip) as z:
    atus_shapefiles = [f for f in z.namelist() if f.lower().endswith(".shp")]
if len(atus_shapefiles) != 1:
    raise ValueError(f"Selecciona la capa ATUS correcta; capas encontradas: {atus_shapefiles}")
atus_shapefiles


In [ ]:
path_mapa = f"zip://{path_atus_zip.as_posix()}!{atus_shapefiles[0]}"
gdf = gpd.read_file(path_mapa)
for column in ["EDO", "MPIO"]:
    gdf[column] = pd.to_numeric(gdf[column], errors="raise")
if gdf.crs is None:
    raise ValueError("ATUS no tiene CRS definido. Revisa que el ZIP contenga el archivo .prj.")
gdf.head()


In [ ]:
gdf.columns.tolist()

In [ ]:
print(gdf.crs)
print(gdf.geometry.geom_type.unique())

In [ ]:
gdf.shape

In [ ]:
# La capa ATUS ya se cargó; no es necesario leerla de nuevo.
print(f"Registros extraídos: {len(gdf)}")
print(f"Columnas: {len(gdf.columns)}")
print(f"CRS: {gdf.crs}")


In [ ]:
gdf[["LATITUD", "LONGITUD", "geometry"]].head()

In [ ]:
gdf[["EDO", "MPIO", "TIPACCID", "ANIO", "MES", "DIA", "HORA"]].head()

In [ ]:
print(ageb_merida.crs)
print(ageb_merida.geometry.geom_type.unique())
print(ageb_merida.geometry.is_valid.all())
print(ageb_merida.shape)
print(ageb_merida["CVE_AGEB"].nunique())

In [ ]:
atus_ageb = gdf.to_crs(ageb_merida.crs)

print(atus_ageb.crs)

In [ ]:
atus_ageb = gpd.sjoin(
    atus_ageb,
    ageb_merida[["CVE_AGEB", "geometry"]],
    how="left",
    predicate="within"
)

print(atus_ageb.shape)
print(atus_ageb["CVE_AGEB"].notna().sum())

In [ ]:
print(gdf["EDO"].value_counts().head())
print(gdf["EDO"].unique())

In [ ]:
print(gdf[["LATITUD", "LONGITUD"]].describe())

In [ ]:
atus_yucatan = gdf[gdf["EDO"] == 31]

print(atus_yucatan["MPIO"].value_counts().sort_index())

In [ ]:
atus_yucatan[["EDO", "MPIO"]].drop_duplicates().sort_values("MPIO")

In [ ]:
atus_merida = gdf[
    (gdf["EDO"] == 31) &
    (gdf["MPIO"] == 50)
].copy()

print("Accidentes en Mérida:", len(atus_merida))

In [ ]:
atus_merida = atus_merida.to_crs(ageb_merida.crs)

atus_merida_ageb = gpd.sjoin(
    atus_merida,
    ageb_merida[["CVE_AGEB", "geometry"]],
    how="left",
    predicate="within"
)

print("Accidentes:", len(atus_merida_ageb))
print("Accidentes asignados a AGEB:", atus_merida_ageb["CVE_AGEB"].notna().sum())

In [ ]:
print("Total de accidentes:", len(atus_merida_ageb))
print("Asignados a AGEB:", atus_merida_ageb["CVE_AGEB"].notna().sum())
print("Sin AGEB:", atus_merida_ageb["CVE_AGEB"].isna().sum())
print(
    "Porcentaje asignado:",
    round(atus_merida_ageb["CVE_AGEB"].notna().mean() * 100, 2),
    "%"
)